In [ ]:
# =============================================================================
# Project:  lv-syn-grid
# Author:   Roman S.
# Created:  2025
#
# Description:
# A pandapower network model is created to enable power flow simulations and
# time series analyses of the generated low-voltage grid.
#
# License: MIT License
#
# =============================================================================

# import for data/math/statistics
import numpy as np
import pandas as pd
import geopandas as gp

# import libraries for power flow analysis
import pandapower as pp
import pandapower.control as control
import pandapower.timeseries as timeseries
from pandapower.timeseries.data_sources.frame_data import DFData
from pandapower.auxiliary import pandapowerNet
import matplotlib.pyplot as plt

# import for file i/o
import json
import pickle

# import utils
from scripts import parameters as prm
from scripts import plot_utils as pu
from scripts import net_utils as nu

In [ ]:
FILE_LOAD_PROFILES = 'synthload2025.xlsx'

# selection 15.01.2025: weekday, winter day
LP_T_START = '2025-01-01T00:00:00'
LP_T_END = '2025-01-01T23:45:00'

# selection 21.06.2025: weekday, summer day
#LP_T_START = '2025-07-15T00:00:00'
#LP_T_END = '2025-07-15T23:45:00'

In [ ]:
class GeoGridModel:
    """
    Builds the model for analyzing low-voltage distribution networks.
    """
    def __init__(self):
        try:
            self.__b_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILE_BUILDINGS_PARQUET)
            self.__s_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILE_STREETS_PARQUET)
            self.__t_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILE_TRANSFORMERS_PARQUET)
            self.__l_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILE_LINES_PARQUET)
            self.__l_gdf['length_km'] = self.__l_gdf['geometry'].apply(lambda line: line.length/1e3)

            #self.__pv_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILE_PV_PARQUET)
        except FileNotFoundError:
            print('File not Found')

        self.__G_lv = None
        self.__H_lv = None
        self.__J_mv = None
        self.__H_s_lv = []

        # create an empty pandapower net
        self.__net = pp.create_empty_network()
        self.__hv_bus = pp.create_bus(self.__net, vn_kv=20., name="HV_BUS", index=0)    # create the hv bus, 20 kV
        pp.create_ext_grid(self.__net, self.__hv_bus, vm_pu=1.0)   # create the external grid for the net

        self.__load_H_s()

        self.__fig_grid_loading = None

        # variables for load profiles
        self.__lp_df = None
        self.__n_ts = 0

        # dataframes for external grid, loads, pv
        self.__ext_grid_df = None
        self.__load_ts_df = None
        self.__pv_ts_df = None
        

        # data frames for simulation results
        self.__simres_trafo_loading_df = None
        self.__simres_line_loading_df = None
        self.__simres_bus_voltage_df = None


    def __load_H_s(self) -> None:
        """
        Loads the network subgraphs from stored file.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        with open(prm.FILEPATH_DATA + 'H_s_lv.pkl', 'rb') as f:
            self.__H_s_lv = pickle.load(f)

    
    def get_H_s_lv(self) -> list:
        """
        Returns a list of the network subgraphs.

        Parameters:
        -----------
        None

        Returns:
        --------
        list
            List of the network subgraphs.
        """
        return self.__H_s_lv
    

    def get_net(self) -> pandapowerNet:
        """
        Returns the pandapower network model.

        Parameters:
        -----------
        None

        Returns:
        --------
        pandapowerNet
            pandapower network representing the electrical grid.
        """
        return self.__net
    

    def create_power_grid(self) -> None:
        """
        Creates the electrical power grid model.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        # create variable transformers with several power values
        self.__net = nu.create_transformers(self.__net)
        
        for G in self.__H_s_lv:
            self.__net = nu.create_lv_busses(G, self.__net, self.__b_gdf)
            self.__net = nu.create_lv_lines(G, self.__net)

        #self.__net = nu.create_lv_busses(self.__H_s_lv[1], self.__net, self.__b_gdf)
        #self.__net = nu.create_lv_lines(self.__H_s_lv[1], self.__net)

        coords = [
            json.loads(g)["coordinates"]
            for g in self.__net.bus["geo"].dropna()
        ]   

        # calculate mean for x, y coordinates
        xs, ys = zip(*coords)
        x_mean = sum(xs) / len(xs)
        y_mean = sum(ys) / len(ys)

        # set geo for HV bus
        self.__net.bus.loc[self.__net.bus["name"] == "HV_BUS", "geo"] = json.dumps({"coordinates": [x_mean, y_mean]})


    def import_load_profiles(self) -> None:
        """
        Imports load profiles for the power grid model.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        lp_df = pd.read_excel(prm.FILEPATH_DATA + prm.FILE_SYNTHETIC_LOAD_PROFILES) # load the load profiles into a dataframe
        lp_df['ts'] = pd.to_datetime(lp_df['ts'])   

        lp_df = lp_df[(lp_df['ts'] >= pd.Timestamp(LP_T_START)) & (lp_df['ts'] <= pd.Timestamp(LP_T_END))].set_index('ts')  # only keep the data within the given time span (ts) and set time step index

        columns_to_keep = ['H0', 'G0', 'G1', 'G2', 'G3', 'G6'] # only keep following load profiles
        columns_to_drop = [col for col in lp_df.columns if col not in columns_to_keep]
        lp_df.drop(columns=columns_to_drop, inplace=True)

        self.__lp_df = lp_df
        self.__n_ts = len(lp_df)


    def calculate_load_timeseries(self) -> None:
        """
        Calculates the load time series for the power grid.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        load_p_kw_arr = []

        # iterate over every load in the net
        for row in self.__net.load.iterrows():
            id = row[1]['name'] # get the id of the net.load dataframe for each load
            load_e_kwh = self.__b_gdf[self.__b_gdf['id'] == id]['energy_demand'].values[0]  # get the energy demand of each load
            slp = self.__b_gdf[self.__b_gdf['id'] == id]['slp'].values[0]   # get the load profile type for each building
            
            load_p_kw = np.reshape((np.array(self.__lp_df[slp]) * load_e_kwh / 1000 / 0.25), (self.__n_ts, 1))
            load_p_kw_arr.append(load_p_kw) # add the power profile to an storage array

        loads_p_mw = np.concatenate(load_p_kw_arr, axis=1)/1e3

        load_ts_df = pd.DataFrame(  loads_p_mw,
                                    index=list(range(self.__n_ts)),
                                    columns=self.__net.load.index)
        
        self.__load_ts_df = load_ts_df


    def get_load_profiles(self) -> pd.DataFrame:
        """
        Returns the load profiles DataFrame.

        Parameters:
        -----------
        None

        Returns:
        --------
        pandas.DataFrame
            DataFrame containing load profiles.
        """
        return self.__lp_df


    def get_load_time_series(self) -> pd.DataFrame:
        """
        Returns the load time series DataFrame.

        Parameters:
        -----------
        None

        Returns:
        --------
        pandas.DataFrame
            DataFrame containing load time series data.
        """
        return self.__load_ts_df


    def implement_pv_systems(self) -> None:
        """
        Implements photovoltaic (PV) systems in the power grid model.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__net.sgen.drop(self.__net.sgen.index, inplace=True)

        self.__pv_gdf = gp.read_parquet(prm.FILEPATH_DATA + 'photovoltaic.parquet')

        for _, row in self.__pv_gdf.iterrows():
            bus = int(self.__net.bus.query(f"node_id == '{row['id']}'").index[0])
        
            sgen_idx = pp.create_sgen(  self.__net,
                                        bus=bus,
                                        p_mw=0.01,
                                        q_mvar=0,
                                        name=f"PV_{row['id']}",
                                        max_p_mw=row["inverter_p_kw"]/1e3)
    
        self.__net.sgen['id'] = self.__pv_gdf['id']
        pv_arr_kw = []

        for row in self.__net.sgen.iterrows():
            id = row[1]['id']
            profile = np.reshape(np.array(self.__pv_gdf[self.__pv_gdf['id'] == id]['pv_profile'].values[0]), ((self.__n_ts, 1)))
            pv_arr_kw.append(profile)

        pv_arr_mw = np.concatenate(pv_arr_kw, axis=1)/1e6
        pv_ts_df = pd.DataFrame(    pv_arr_mw,
                                    index=list(range(self.__n_ts)),
                                    columns=self.__net.sgen.index)
        
        self.__pv_ts_df = pv_ts_df


    def create_controller(self) -> None:
        """
        Creates controllers for time series power flow analysis.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        ext_grid_df = pd.DataFrame(  np.reshape(np.ones((self.__n_ts, 1))*1, (self.__n_ts, 1)),
                                        index=list(range(self.__n_ts)),
                                        columns=self.__net.ext_grid.index)
        
        # create the controller for the external grid
        const_ext_grid = control.ConstControl(  self.__net,
                                                element='ext_grid',
                                                element_index=self.__net.ext_grid.index,
                                                variable='p_mw', 
                                                data_source=DFData(ext_grid_df), 
                                                profile_name=self.__net.ext_grid.index   )
        
        if self.__load_ts_df is not None:
            # create the controller for the loads
            const_load = control.ConstControl(  self.__net,
                                                element='load',
                                                element_index=self.__net.load.index,
                                                variable='p_mw',
                                                data_source=DFData(self.__load_ts_df),
                                                profile_name=self.__net.load.index)
        
        if self.__pv_ts_df is not None:
            # create the controller for the pv systems
            const_sgen = control.ConstControl(  self.__net,
                                                element='sgen',
                                                element_index=self.__net.sgen.index,
                                                variable='p_mw',
                                                data_source=DFData(self.__pv_ts_df),
                                                profile_name=self.__net.sgen.index)

    
    def create_output_writer(self) -> None:
        """
        Creates an output writer for storing time series simulation results.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        # initialising the outputwriter to save data to excel files in the current folder (.xlsx, .json, .csv, or .pickle)
        ow = timeseries.OutputWriter(self.__net, output_path=prm.FILEPATH_SIMRES, output_file_type=".xlsx")
        
        # adding vm_pu of all buses and line_loading in percent of all lines as outputs to be stored
        ow.log_variable('res_bus', 'vm_pu') # log vm_pu (relative voltage) of all busses
        ow.log_variable('res_line', 'loading_percent')  # log the loading of all lines [%]
        ow.log_variable('res_trafo', 'loading_percent') # log the loading of all trafos [%]
        #ow.log_variable('res_line', 'i_from_ka')
        #ow.log_variable('res_line', 'i_to_ka')
        #print(ow.log_variables)


    def run_simulation(self) -> None:
        """
        Runs the power flow time series simulation.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        timeseries.run_timeseries(self.__net,
                                  time_steps=list(range(self.__n_ts)))


    def save_net(self, filename: str) -> None:
        """
        Saves the pandapower network model to disk.

        Parameters:
        -----------
        filename : str
            Base filename used for saving the network model.

        Returns:
        --------
        None
        """
        pp.to_pickle(self.__net, f'{prm.FILEPATH_LV_GRID}{filename}.p')

    
    def set_net(self, net) -> None:
        """
        Sets the pandapower network model.

        Parameters:
        -----------
        net : pandapowerNet
            pandapower network to be assigned.

        Returns:
        --------
        None
        """
        self.__net = net


    def load_net(self, filename: str) -> None:
        """
        Loads a pandapower network model from file.

        Parameters:
        -----------
        filename : str
            Base filename of the network model to be loaded.

        Returns:
        --------
        None
        """
        self.__net = pp.from_pickle(prm.FILEPATH_LV_GRID + f'{filename}.p')

    
    def read_sim_results(self) -> None:
        """
        Reads and stores time series simulation results from files.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        simres_trafo_loading_df = pd.read_excel(prm.FILEPATH_SIMRES + prm.FILE_SIMRES_TRAFOLOADING, sheet_name=0).drop(columns=['Unnamed: 0'], errors='ignore')
        simres_trafo_loading_df.index = pd.date_range(LP_T_START, periods=96, freq="15min")

        simres_line_loading_df = pd.read_excel(prm.FILEPATH_SIMRES + prm.FILE_SIMRES_LINELOADING, sheet_name=0).drop(columns=["Unnamed: 0"], errors="ignore")
        simres_line_loading_df.index = pd.date_range(LP_T_START, periods=96, freq="15min")

        simres_bus_voltage_df = pd.read_excel(prm.FILEPATH_SIMRES + prm.FILE_SIMRES_BUSVOLTAGE, sheet_name=0).drop(columns=["Unnamed: 0"], errors="ignore")
        simres_bus_voltage_df.index = pd.date_range(LP_T_START, periods=96, freq="15min")

        self.__simres_trafo_loading_df = simres_trafo_loading_df
        self.__simres_line_loading_df = simres_line_loading_df
        self.__simres_bus_voltage_df = simres_bus_voltage_df

        
    def plot_grid_loading_osm(self) -> None:
        """
        Plots the grid loading and bus voltages on a basemap.

        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__fig_grid_loading = pu.plot_grid_loading_osm( net=self.__net,
                                                            simres_line_loading_df=self.__simres_line_loading_df,
                                                            simres_bus_voltage_df=self.__simres_bus_voltage_df,
                                                            s_gdf=self.__s_gdf)

        
    def save_fig_grid_loading(self, filename: str) -> None:
        """
        Saves the grid loading visualization to files.

        Parameters:
        -----------
        filename : str
            Base filename used for saving the figure.

        Returns:
        --------
        None
        """
        self.__fig_grid_loading.savefig(f'{prm.FILEPATH_PLOTS}{filename}.svg', format='svg', dpi=600, bbox_inches='tight', transparent=True)
        self.__fig_grid_loading.savefig(f'{prm.FILEPATH_PLOTS}{filename}.png', format='png', dpi=600, bbox_inches='tight', transparent=True)

In [ ]:
grm = GeoGridModel()

In [ ]:
grm.get_load_time_series()

In [ ]:
grm.create_power_grid()
#grm.load_net(filename='net')

In [ ]:
grm.import_load_profiles()
grm.calculate_load_timeseries()
#grm.implement_pv_systems()
grm.create_controller()
grm.create_output_writer()

In [ ]:
grm.save_net('net')

In [ ]:
grm.run_simulation()

In [ ]:
grm.read_sim_results()
grm.plot_grid_loading_osm()

In [ ]:
# save the grid loading image as image file
grm.save_fig_grid_loading(filename='grid_loading')